In [1]:
import hyperopt

print(hyperopt.__version__)

0.2.7


In [2]:
from hyperopt import hp

search_space = {'x' : hp.quniform('x', -10, 10, 1),  'y' : hp.quniform('y', -15, 15, 1)} # -10에서 10까지 1씩 증가


In [3]:
from hyperopt import STATUS_OK

def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y # 목적함수
    return retval

In [4]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

best_01 = fmin(fn = objective_func, space= search_space, algo= tpe.suggest, max_evals= 5, trials= trial_val, rstate = np.random.default_rng(seed = 0))
print(best_01)

100%|██████████| 5/5 [00:00<00:00, 948.21trial/s, best loss: -224.0]
{'x': -4.0, 'y': 12.0}


In [5]:
trial_val = Trials()

best_02 = fmin(fn = objective_func, space= search_space, algo= tpe.suggest, max_evals= 20, trials= trial_val, rstate = np.random.default_rng(seed = 0))
print(best_02)

100%|██████████| 20/20 [00:00<00:00, 1732.04trial/s, best loss: -296.0]
{'x': 2.0, 'y': 15.0}


In [6]:
print(trial_val.results)

[{'loss': -64.0, 'status': 'ok'}, {'loss': -184.0, 'status': 'ok'}, {'loss': 56.0, 'status': 'ok'}, {'loss': -224.0, 'status': 'ok'}, {'loss': 61.0, 'status': 'ok'}, {'loss': -296.0, 'status': 'ok'}, {'loss': -40.0, 'status': 'ok'}, {'loss': 281.0, 'status': 'ok'}, {'loss': 64.0, 'status': 'ok'}, {'loss': 100.0, 'status': 'ok'}, {'loss': 60.0, 'status': 'ok'}, {'loss': -39.0, 'status': 'ok'}, {'loss': 1.0, 'status': 'ok'}, {'loss': -164.0, 'status': 'ok'}, {'loss': 21.0, 'status': 'ok'}, {'loss': -56.0, 'status': 'ok'}, {'loss': 284.0, 'status': 'ok'}, {'loss': 176.0, 'status': 'ok'}, {'loss': -171.0, 'status': 'ok'}, {'loss': 0.0, 'status': 'ok'}]


In [7]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer()
df = pd.DataFrame(data= dataset.data, columns=dataset.feature_names)
df['target'] = dataset.target

X_features = df.iloc[:, :-1]
y_label = df.iloc[:, -1]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_features, y_label, test_size=0.2, random_state=156)

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=156)

In [9]:
from hyperopt import hp

xgb_search_space = {
    'max_depth' :hp.quniform('max_depth', 5, 20, 1),
    'min_child_weight' :hp.quniform('min_child_weight', 1, 2, 1),
    'learning_rate' :hp.uniform('learning_rate', 0.01, 0.2),
    'colsample_bytree' :hp.uniform('colsample_bytree', 0.5, 1)
}

In [14]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators = 100,
                            max_depth = int(search_space['max_depth']),
                            min_child_weight= int(search_space['min_child_weight']),
                            learning_rate= int(search_space['learning_rate']),
                            colsample_bytree= int(search_space['colsample_bytree']),
                            eval_metric = 'logloss')
    accuracy = cross_val_score(xgb_clf, X_train, y_train,
                               scoring='accuracy', cv=3)
    return {'loss': -1* np.mean(accuracy), 'status':STATUS_OK}

In [15]:
trial_val = Trials()
best = fmin(fn = objective_func, space= xgb_search_space, algo = tpe.suggest, max_evals= 50,
            trials= trial_val, rstate=np.random.default_rng(42))
print(best)

100%|██████████| 50/50 [00:09<00:00,  5.16trial/s, best loss: -0.3846142674567213]
{'colsample_bytree': 0.825871199874447, 'learning_rate': 0.17794605883979694, 'max_depth': 14.0, 'min_child_weight': 2.0}


In [16]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred = None, pred_proba= None) :
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc_auc = roc_auc_score(y_test, pred_proba)

    print('오차행렬')
    print(confusion)
    print('정확도 :', accuracy)
    print('정밀도 :', precision)
    print('재현율 :', recall)
    print('f1 :', f1)
    print('roc_auc :', roc_auc)

In [18]:
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate= round(best['learning_rate'],5),
                            max_depth= int(best['max_depth']),
                            min_child_weight= int(best['min_child_weight']),
                            colsample_bytree= round(best['colsample_bytree'],5))

evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, early_stopping_rounds= 50,
                eval_metric = 'logloss', eval_set = evals, verbose =True)
preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]
get_clf_eval(y_test, preds, pred_proba)

[0]	validation_0-logloss:0.55054	validation_1-logloss:0.59399
[1]	validation_0-logloss:0.44820	validation_1-logloss:0.51546
[2]	validation_0-logloss:0.37308	validation_1-logloss:0.46742
[3]	validation_0-logloss:0.31667	validation_1-logloss:0.44066
[4]	validation_0-logloss:0.26839	validation_1-logloss:0.41505
[5]	validation_0-logloss:0.23027	validation_1-logloss:0.38322
[6]	validation_0-logloss:0.20034	validation_1-logloss:0.35666
[7]	validation_0-logloss:0.17650	validation_1-logloss:0.33748
[8]	validation_0-logloss:0.15400	validation_1-logloss:0.32286
[9]	validation_0-logloss:0.13562	validation_1-logloss:0.31151
[10]	validation_0-logloss:0.12070	validation_1-logloss:0.30455
[11]	validation_0-logloss:0.10904	validation_1-logloss:0.29557
[12]	validation_0-logloss:0.09753	validation_1-logloss:0.29321
[13]	validation_0-logloss:0.08690	validation_1-logloss:0.29002
[14]	validation_0-logloss:0.07962	validation_1-logloss:0.29044
[15]	validation_0-logloss:0.07186	validation_1-logloss:0.28684
[1